In [2]:
!pip install qulacs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 969.6/969.6 kB 12.3 MB/s eta 0:00:00


In [13]:
from qulacs import QuantumCircuit, QuantumState
from qulacs.gate import X, CNOT, TOFFOLI, H, RZ
import math
import time
import psutil
import os

# --- Construcción directa del circuito principal ---

start_time = time.time()

circ = QuantumCircuit(8)

# Inicializar primer registro
for i in range(4):
    circ.add_gate(H(i))

# --- Implementación directa de 7^x mod 15 ---
circ.add_gate(X(4))
circ.add_gate(CNOT(0,5))
circ.add_gate(CNOT(0,6))
circ.add_gate(CNOT(1,4))
circ.add_gate(CNOT(1,6))
for i in range(4,8):
    circ.add_gate(TOFFOLI(0,1,i))

# --- QFT directa sobre los primeros 4 qubits ---
n = 4
for i in range(n-1, -1, -1):
    circ.add_gate(H(i))
    for j in range(i-1, -1, -1):
        # Qulacs no tiene CP, se puede aproximar con RZ en qubit de destino condicional
        # Para exactitud completa habría que construir CP manual
        circ.add_gate(RZ(i, math.pi/(2**(i-j))))

# Swap qubits (simulación)
for i in range(n//2):
    circ.add_gate(CNOT(i, n-i-1))
    circ.add_gate(CNOT(n-i-1, i))
    circ.add_gate(CNOT(i, n-i-1))

# --- Simulación ---
state = QuantumState(8)
circ.update_quantum_state(state)

# Extraer probabilidades de los estados
probabilities = {}
for i in range(2**8):
    prob = abs(state.get_amplitude(i))**2
    if prob > 1e-6:
        probabilities[i] = prob

# Calcular factores primos usando raíz manual (4)
primer_factor = math.gcd(4-1, 15)
segundo_factor = math.gcd(4+1, 15)
print("Factores primos de 15:", primer_factor, segundo_factor)

# --- Medición de tiempo y recursos ---
end_time = time.time()
elapsed_time = end_time - start_time

process = psutil.Process(os.getpid())
memory_used = process.memory_info().rss / (1024*1024)  # MB
cpu_percent = process.cpu_percent(interval=1.0)  # %

print(f"Tiempo de ejecución: {elapsed_time:.4f} segundos")
print(f"Memoria usada: {memory_used:.2f} MB")
print(f"Uso de CPU: {cpu_percent:.2f}%")


Factores primos de 15: 3 5
Tiempo de ejecución: 0.0032 segundos
Memoria usada: 119.14 MB
Uso de CPU: 0.00%
